# Alphabet 위임장(DEF 14A) 안건 목록 추출
SEC에서 최신 DEF 14A를 받아 안건 번호·제목·제안 주체·이사회 권고·쪽번호·유형을 정리한다.

출력 파일:
- `data/raw/googl/def14a_<연도>.htm` — 위임장 원본 HTML
- `data/processed/googl_agenda_<연도>.csv` — 안건 목록

## 1. 환경 준비
edgartools를 설치하고 `proxy` 객체의 속성을 확인한다(`proxy`는 이 노트북에서 정의되지 않음).

In [9]:
#!pip install beautifulsoup4 lxml
!pip install edgartools

In [16]:
print(proxy)   # 객체가 요약해서 보여주는 내용

attrs = [a for a in dir(proxy) if not a.startswith("_")]
print(f"\n사용 가능한 속성·기능 {len(attrs)}개:")
print(attrs)

DEF 14A: Alphabet Inc. - 2026-04-24

사용 가능한 속성·기능 40개:
['accession_number', 'audit_fees', 'award_dates_predetermined', 'award_timing_mnpi_considered', 'awards_close_to_mnpi', 'beneficial_ownership', 'ceo_pay_ratio', 'cik', 'company_name', 'company_selected_measure', 'company_selected_measure_value', 'director_compensation_table', 'document', 'executive_compensation', 'filing', 'filing_date', 'fiscal_year_end', 'form', 'from_filing', 'has_individual_executive_data', 'has_xbrl', 'insider_trading_policy_adopted', 'mnpi_disclosure_timed_for_comp_value', 'named_executives', 'neo_avg_actually_paid_comp', 'neo_avg_total_comp', 'net_income', 'pay_vs_performance', 'peer_group_tsr', 'peo_actually_paid_comp', 'peo_name', 'peo_total_comp', 'performance_measures', 'season', 'section', 'sections', 'summary_compensation_table', 'to_context', 'total_shareholder_return', 'voting_proposals']


## 2. 위임장 다운로드와 안건 제목 추출
티커 → CIK → 최신 DEF 14A를 받고, HTML 태그에서 'Proposal N' 제목과 제안 주체를 뽑는다.

In [17]:
import re
import time
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import requests
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

# ===== 설정 =====
HEADERS = {"User-Agent": "Your Name your_email@example.com"}   # 본인 이름·이메일로
TICKER = "GOOGL"          # 다른 회사: "ADBE", "AAPL", "NVDA" 등
TARGET_YEAR = None        # None이면 가장 최근 위임장, 특정 연도는 예: 2025 (제출 연도 기준)


def sec_get(url):
    """SEC 요청 (초당 요청 제한을 지키려고 잠깐 쉼)"""
    r = requests.get(url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    time.sleep(0.15)
    return r


# ===== 1. 티커 → CIK =====
tickers = sec_get("https://www.sec.gov/files/company_tickers.json").json()
row = next(v for v in tickers.values() if v["ticker"] == TICKER)
cik = int(row["cik_str"])
cik10 = f"{cik:010d}"
print(f"[1] {TICKER} → {row['title']} (CIK {cik})")

# ===== 2. 공시 목록에서 DEF 14A 고르기 =====
sub = sec_get(f"https://data.sec.gov/submissions/CIK{cik10}.json").json()
recent = pd.DataFrame(sub["filings"]["recent"])
proxies = recent[recent["form"] == "DEF 14A"].copy()
proxies["filingDate"] = pd.to_datetime(proxies["filingDate"])
if TARGET_YEAR:
    proxies = proxies[proxies["filingDate"].dt.year == TARGET_YEAR]
if proxies.empty:
    raise ValueError("조건에 맞는 DEF 14A가 없어요. TARGET_YEAR를 확인하세요.")

p = proxies.sort_values("filingDate").iloc[-1]
acc, doc = p["accessionNumber"], p["primaryDocument"]
url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc.replace('-', '')}/{doc}"
print(f"[2] 제출일 {p['filingDate'].date()} / 주총일(reportDate) {p['reportDate']} / {doc}")

# ===== 3. 다운로드 (이미 있으면 재사용) =====
path = Path(f"data/raw/{TICKER.lower()}/def14a_{p['filingDate'].year}.htm")
path.parent.mkdir(parents=True, exist_ok=True)
if not path.exists():
    path.write_bytes(sec_get(url).content)
print(f"[3] 파일: {path} ({path.stat().st_size:,} bytes)")

# ===== 4. 화면에 보이는 본문 글자만 꺼내기 =====
soup = BeautifulSoup(path.read_text(encoding="utf-8", errors="ignore"), "lxml")
for h in soup.find_all(style=re.compile(r"display:\s*none", re.I)):
    h.decompose()
text = re.sub(r"\s+", " ", soup.body.get_text(" ", strip=True))
print(f"[4] 본문 글자 수: {len(text):,}")

# ===== 5. (수정판) HTML 태그 단위로 안건 제목 찾기 =====
heading_pat = re.compile(
    r"(?i)^proposal\s*(?:no\.?|number)?\s*(\d{1,2})\b\s*[—–:\-.]?\s*(.*)$"
)

candidates = defaultdict(list)
for el in soup.body.find_all(["p", "div", "span", "td", "b", "strong", "font", "h1", "h2", "h3", "h4"]):
    t = re.sub(r"\s+", " ", el.get_text(" ", strip=True))
    if len(t) > 250:                       # 긴 문단은 제목이 아님
        continue
    m = heading_pat.match(t)
    if not m:
        continue
    title = re.sub(r"\s\d{1,3}$", "", m.group(2)).strip(" —–:-.")   # 끝의 쪽번호 제거
    if len(title) >= 8:
        candidates[int(m.group(1))].append(title)

# 1번부터 연속된 번호만 안건으로 인정
proposal_nums = []
n = 1
while n in candidates:
    proposal_nums.append(n)
    n += 1

def pick_title(titles):
    """가장 자주 나온 제목, 횟수가 같으면 더 짧은 것"""
    counts = Counter(titles)
    return sorted(counts.items(), key=lambda x: (-x[1], len(x[0])))[0]

rows = []
for num in proposal_nums:
    title, freq = pick_title(candidates[num])
    is_holder = re.match(r"(?i)^(stockholder|shareholder)\s+proposal", title) is not None
    rows.append({
        "proposal_no": num,
        "title": title,
        "proposer": "주주" if is_holder else "회사",
        "mentions": len(candidates[num]),
        "title_freq": freq,
    })

agenda = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 150)
print(f"[5] 안건 수: {len(agenda)}개 (회사 {sum(agenda.proposer == '회사')}, 주주 {sum(agenda.proposer == '주주')})")
agenda

[1] GOOGL → Alphabet Inc. (CIK 1652044)
[2] 제출일 2026-04-24 / 주총일(reportDate) 2026-06-05 / goog-20260424.htm
[3] 파일: data\raw\googl\def14a_2026.htm (2,175,852 bytes)
[4] 본문 글자 수: 403,020
[5] 안건 수: 14개 (회사 4, 주주 10)


,proposal_no,title,proposer,mentions,title_freq
0,1,Election of Directors,회사,5,5
1,2,Ratification of Appointment of Independent Registered Public Accounting Firm,회사,5,5
2,3,A pproval of the Amendment and Restatement of Alphabet Inc. Amended and Restated 2021 Stock Plan,회사,5,3
3,4,Advisory Vote to Approve Compensation Awarded to Named Executive Officers,회사,5,5
4,5,Shareholder Proposal Regarding an Enhanced Disclosure on Climate Goals,주주,5,5
5,6,Shareholder Proposal Regarding a Report on Water Usage and AI Development,주주,5,5
6,7,Shareholder Proposal Regarding Equal Shareholder Voting,주주,5,5
7,8,Shareholder Proposal Regarding a Viewpoint Diversity Risk Report,주주,5,4
8,9,Shareholder Proposal Regarding a Report on Politicized Content Moderation,주주,5,4
9,10,Shareholder Proposal Regarding a Report on Impact of U.S. Immigration Policy,주주,5,5


## 3. 이사회 권고 찾기(1차)와 저장
제목 뒤 2,000자 안의 'recommends … FOR/AGAINST'로 권고를 찾고 유형을 붙여 CSV로 저장한다.

In [18]:
rec_pat = re.compile(r"(?i)recommends?\s+(?:that\s+(?:you|stockholders|shareholders)\s+vote\s+|a\s+vote\s+)?[\"“”]?(FOR|AGAINST)\b")

def find_board_rec(title, window=2000):
    """제목이 나오는 곳마다 뒤쪽 window자 안에서 이사회 권고를 찾고, 가장 많이 나온 값을 고름"""
    found = []
    start = 0
    while True:
        i = text.find(title, start)
        if i == -1:
            break
        m = rec_pat.search(text, i, i + window)
        if m:
            found.append(m.group(1).upper())
        start = i + len(title)
    if not found:
        return None, 0
    value, cnt = Counter(found).most_common(1)[0]
    return value, len(found)

recs = agenda["title"].apply(find_board_rec)
agenda["board_rec"] = recs.str[0]
agenda["rec_hits"] = recs.str[1]

agenda[["proposal_no", "title", "proposer", "board_rec", "rec_hits"]]

,proposal_no,title,proposer,board_rec,rec_hits
0,1,Election of Directors,회사,None,0
1,2,Ratification of Appointment of Independent Registered Public Accounting Firm,회사,None,0
2,3,A pproval of the Amendment and Restatement of Alphabet Inc. Amended and Restated 2021 Stock Plan,회사,None,0
3,4,Advisory Vote to Approve Compensation Awarded to Named Executive Officers,회사,None,0
4,5,Shareholder Proposal Regarding an Enhanced Disclosure on Climate Goals,주주,None,0
5,6,Shareholder Proposal Regarding a Report on Water Usage and AI Development,주주,None,0
6,7,Shareholder Proposal Regarding Equal Shareholder Voting,주주,None,0
7,8,Shareholder Proposal Regarding a Viewpoint Diversity Risk Report,주주,None,0
8,9,Shareholder Proposal Regarding a Report on Politicized Content Moderation,주주,None,0
9,10,Shareholder Proposal Regarding a Report on Impact of U.S. Immigration Policy,주주,None,0


In [19]:
def agenda_type(row):
    t = row["title"].lower()
    if row["proposer"] == "주주":
        return "주주제안"
    if "election of director" in t:
        return "이사 선임"
    if "accounting firm" in t or "auditor" in t:
        return "회계법인 선임"
    if "compensation" in t and ("advisory" in t or "named executive" in t):
        return "경영진 보수"
    if "stock plan" in t or "incentive plan" in t or "equity" in t:
        return "주식보상계획"
    if "frequency" in t:
        return "보수 투표 주기"
    if "certificate" in t or "bylaw" in t or "charter" in t:
        return "정관 변경"
    return "기타"

agenda["agenda_type"] = agenda.apply(agenda_type, axis=1)

out_csv = Path(f"data/processed/{TICKER.lower()}_agenda_{p['filingDate'].year}.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)
agenda.to_csv(out_csv, index=False, encoding="utf-8-sig")
print("저장 완료:", out_csv)

agenda[["proposal_no", "title", "proposer", "board_rec", "agenda_type"]]

저장 완료: data\processed\googl_agenda_2026.csv


,proposal_no,title,proposer,board_rec,agenda_type
0,1,Election of Directors,회사,None,이사 선임
1,2,Ratification of Appointment of Independent Registered Public Accounting Firm,회사,None,회계법인 선임
2,3,A pproval of the Amendment and Restatement of Alphabet Inc. Amended and Restated 2021 Stock Plan,회사,None,주식보상계획
3,4,Advisory Vote to Approve Compensation Awarded to Named Executive Officers,회사,None,경영진 보수
4,5,Shareholder Proposal Regarding an Enhanced Disclosure on Climate Goals,주주,None,주주제안
5,6,Shareholder Proposal Regarding a Report on Water Usage and AI Development,주주,None,주주제안
6,7,Shareholder Proposal Regarding Equal Shareholder Voting,주주,None,주주제안
7,8,Shareholder Proposal Regarding a Viewpoint Diversity Risk Report,주주,None,주주제안
8,9,Shareholder Proposal Regarding a Report on Politicized Content Moderation,주주,None,주주제안
9,10,Shareholder Proposal Regarding a Report on Impact of U.S. Immigration Policy,주주,None,주주제안


## 4. 권고 점검과 요약표 기반 재추출
'recommend' 문맥을 확인한 뒤 요약표의 '번호 제목 (page N) FOR/AGAINST' 패턴으로 권고·쪽번호를 다시 뽑는다.

In [20]:
# 1) 제목이 본문에 몇 번 나오는지
agenda["title_found"] = agenda["title"].apply(lambda t: text.count(t))
print(agenda[["proposal_no", "title_found"]].to_string(index=False), "\n")

# 2) 'recommend'가 들어간 문장이 실제로 어떻게 생겼는지 (처음 12개)
hits = list(re.finditer(r"(?i)recommend", text))
print(f"'recommend' 등장 횟수: {len(hits)}\n")
for m in hits[:12]:
    print(f"[{m.start():,}] ...{text[max(0, m.start()-80): m.start()+120]}...\n")

 proposal_no  title_found
           1            2
           2            2
           3            1
           4            2
           5            2
           6            2
           7            2
           8            2
           9            2
          10            2
          11            2
          12            2
          13            2
          14            2 

'recommend' 등장 횟수: 87

[7,024] ...business on April 6, 2026 (Record Date) Items of Business Alphabet Board Voting Recommendation 1. Election of directors: Larry Page, Sergey Brin, Sundar Pichai, John L. Hennessy, Frances H. Arnold, R....

[29,843] ...utive Compensation Audit Matters Proposals Q&A Appendix Voting Matters and Vote Recommendations Proposal Alphabet Board Voting Recommendation Rationale Management Proposals: 1 Election of ten director...

[29,890] ...Appendix Voting Matters and Vote Recommendations Proposal Alphabet Board Voting Recommendation Rationale Management Proposals: 1 Election of

In [21]:
summary_pat = re.compile(
    r"(?<!\d)(\d{1,2})\s+([A-Z][^()]{5,250}?)\(\s*page\s*(\d{1,3})\s*\)\s*(FOR|AGAINST)\b"
)

rec_map, page_map = {}, {}
for m in summary_pat.finditer(text):
    num = int(m.group(1))
    if num in agenda["proposal_no"].values and num not in rec_map:   # 처음 나온 것만 사용
        rec_map[num] = m.group(4)
        page_map[num] = int(m.group(3))

agenda["board_rec"] = agenda["proposal_no"].map(rec_map)
agenda["page"] = agenda["proposal_no"].map(page_map)

print(f"권고를 찾은 안건: {agenda['board_rec'].notna().sum()} / {len(agenda)}")
agenda[["proposal_no", "title", "proposer", "board_rec", "page"]]

권고를 찾은 안건: 14 / 14


,proposal_no,title,proposer,board_rec,page
0,1,Election of Directors,회사,FOR,61
1,2,Ratification of Appointment of Independent Registered Public Accounting Firm,회사,FOR,62
2,3,A pproval of the Amendment and Restatement of Alphabet Inc. Amended and Restated 2021 Stock Plan,회사,FOR,63
3,4,Advisory Vote to Approve Compensation Awarded to Named Executive Officers,회사,FOR,68
4,5,Shareholder Proposal Regarding an Enhanced Disclosure on Climate Goals,주주,AGAINST,72
5,6,Shareholder Proposal Regarding a Report on Water Usage and AI Development,주주,AGAINST,74
6,7,Shareholder Proposal Regarding Equal Shareholder Voting,주주,AGAINST,77
7,8,Shareholder Proposal Regarding a Viewpoint Diversity Risk Report,주주,AGAINST,80
8,9,Shareholder Proposal Regarding a Report on Politicized Content Moderation,주주,AGAINST,83
9,10,Shareholder Proposal Regarding a Report on Impact of U.S. Immigration Policy,주주,AGAINST,85


## 5. 제목 정리 후 최종 저장
제목 띄어쓰기를 고치고 유형을 다시 붙여 안건 CSV를 덮어쓴다.

In [22]:
# 1) 제목 띄어쓰기 정리: "A pproval" → "Approval"
#    (대문자 한 글자 + 공백 + 소문자로 이어지는 경우만 붙임)
def fix_title(t):
    return re.sub(r"\b([A-Z]) ([a-z]{2,})", r"\1\2", t)

before = agenda["title"].copy()
agenda["title"] = agenda["title"].apply(fix_title)

changed = agenda.loc[before != agenda["title"], ["proposal_no", "title"]]
print("띄어쓰기를 고친 제목:")
print(changed.to_string(index=False) if len(changed) else "없음", "\n")


# 2) 안건 유형 붙이기 (제목 키워드 기준)
def agenda_type(row):
    t = row["title"].lower()
    if row["proposer"] == "주주":
        return "주주제안"
    if "election of director" in t:
        return "이사 선임"
    if "accounting firm" in t or "auditor" in t:
        return "회계법인 선임"
    if "compensation" in t and ("advisory" in t or "named executive" in t):
        return "경영진 보수"
    if "stock plan" in t or "incentive plan" in t or "equity" in t:
        return "주식보상계획"
    if "frequency" in t:
        return "보수 투표 주기"
    if "certificate" in t or "bylaw" in t or "charter" in t:
        return "정관 변경"
    return "기타"

agenda["agenda_type"] = agenda.apply(agenda_type, axis=1)


# 3) 저장
cols = ["proposal_no", "title", "proposer", "board_rec", "page", "agenda_type"]
out_csv = Path(f"data/processed/{TICKER.lower()}_agenda_{p['filingDate'].year}.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)
agenda[cols].to_csv(out_csv, index=False, encoding="utf-8-sig")
print("저장 완료:", out_csv)

agenda[cols]

띄어쓰기를 고친 제목:
 proposal_no                                                                                           title
           3 Approval of the Amendment and Restatement of Alphabet Inc. Amended and Restated 2021 Stock Plan 

저장 완료: data\processed\googl_agenda_2026.csv


,proposal_no,title,proposer,board_rec,page,agenda_type
0,1,Election of Directors,회사,FOR,61,이사 선임
1,2,Ratification of Appointment of Independent Registered Public Accounting Firm,회사,FOR,62,회계법인 선임
2,3,Approval of the Amendment and Restatement of Alphabet Inc. Amended and Restated 2021 Stock Plan,회사,FOR,63,주식보상계획
3,4,Advisory Vote to Approve Compensation Awarded to Named Executive Officers,회사,FOR,68,경영진 보수
4,5,Shareholder Proposal Regarding an Enhanced Disclosure on Climate Goals,주주,AGAINST,72,주주제안
5,6,Shareholder Proposal Regarding a Report on Water Usage and AI Development,주주,AGAINST,74,주주제안
6,7,Shareholder Proposal Regarding Equal Shareholder Voting,주주,AGAINST,77,주주제안
7,8,Shareholder Proposal Regarding a Viewpoint Diversity Risk Report,주주,AGAINST,80,주주제안
8,9,Shareholder Proposal Regarding a Report on Politicized Content Moderation,주주,AGAINST,83,주주제안
9,10,Shareholder Proposal Regarding a Report on Impact of U.S. Immigration Policy,주주,AGAINST,85,주주제안
